# Meta-Analysis: Pooling Studies — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-1/meta-analysis.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-1/meta-analysis.ipynb)

> Companion to the **Meta-Analysis** article. Pool four odds-ratio studies **two ways** — **fixed-effect** and **DerSimonian–Laird random-effects** — and watch inverse-variance weighting, Cochran's Q, I², and τ² appear one number at a time.

## What you'll build

1. `meta_analyze` — inverse-variance **fixed-effect** and **random-effects** pools with 95% CIs.
2. The heterogeneity trio: **Cochran's Q**, **I²**, and **τ²** (between-study variance).
3. A run that reproduces the article's pooled ORs and the fixed→random **weight shift** — pure `numpy`, **no API key**.

### Setup (numpy only)

In [ ]:
import numpy as np

Z = 1.96   # 95% two-sided normal multiplier (classic constant)
print("ready — Z =", Z)

## The four studies

Four RCTs of the same blood-pressure drug across four countries. Each reports an **odds ratio**; ratios are skewed, so we pool on the **log scale** (`θ = ln(OR)`) and exponentiate at the end. We use the 2-decimal `ln(OR)` values from the article's worked table so the printed numbers match verbatim.

| Study | OR | θ = ln(OR) | SE | w = 1/SE² |
|---|---|---|---|---|
| 1 | 0.60 | −0.51 | 0.20 | 25.00 |
| 2 | 0.70 | −0.36 | 0.25 | 16.00 |
| 3 | 0.40 | −0.92 | 0.30 | 11.11 |
| 4 | 0.90 | −0.11 | 0.15 | 44.44 |

In [ ]:
ORs   = np.array([0.60, 0.70, 0.40, 0.90])
theta = np.array([-0.51, -0.36, -0.92, -0.11])   # ln(OR), 2-dp (matches the article table)
se    = np.array([0.20, 0.25, 0.30, 0.15])        # SE on the log scale
w     = 1.0 / se**2                               # inverse-variance weights
print("weights w = 1/SE^2 :", np.round(w, 2))
print("precise study (small SE) -> big weight; study 4 dominates at", round(w[3],2))

## The pooling function

**Fixed-effect** assumes one true effect (`w = 1/SE²`). **Random-effects** admits the true effects *vary* and adds between-study variance **τ²** to every weight (`w* = 1/(SE² + τ²)`), which widens the CI and pulls the weights toward equal. τ² comes from **Cochran's Q** via DerSimonian–Laird.

In [ ]:
def meta_analyze(effects, ses):
    """Fixed-effect and DL random-effects pools on the analysis scale (log scale for ratios)."""
    theta = np.asarray(effects, float)
    se = np.asarray(ses, float)
    w = 1.0 / se**2
    k = len(theta)

    # ---- fixed-effect pool ----
    theta_fixed = np.sum(w * theta) / np.sum(w)
    se_fixed = np.sqrt(1.0 / np.sum(w))

    # ---- heterogeneity: Cochran's Q, I2, DL tau^2 ----
    Q = np.sum(w * (theta - theta_fixed)**2)
    df = k - 1
    I2 = max(0.0, (Q - df) / Q) * 100 if Q > 0 else 0.0
    denom = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (Q - df) / denom) if denom > 0 else 0.0

    # ---- random-effects pool (weights include tau^2) ----
    w_star = 1.0 / (se**2 + tau2)
    theta_random = np.sum(w_star * theta) / np.sum(w_star)
    se_random = np.sqrt(1.0 / np.sum(w_star))

    def ci(t, s):
        return t - Z*s, t + Z*s

    return {"fixed":  (theta_fixed,  *ci(theta_fixed,  se_fixed),  w / w.sum()),
            "random": (theta_random, *ci(theta_random, se_random), w_star / w_star.sum()),
            "Q": Q, "df": df, "I2": I2, "tau2": tau2}

print("meta_analyze defined")

### Run it — fixed, then random, then the heterogeneity trio

In [ ]:
r = meta_analyze(theta, se)
for model in ("fixed", "random"):
    est, lo, hi, weights = r[model]
    print(f"{model:6s}: OR = {np.exp(est):.3f}  "
          f"(95% CI {np.exp(lo):.3f} to {np.exp(hi):.3f})  "
          f"weights % = {np.round(weights*100, 1)}")
print(f"Q = {r['Q']:.2f}  df = {r['df']}  I2 = {r['I2']:.1f}%  tau2 = {r['tau2']:.4f}")

## Watch the weight shift

Switching from fixed to random did two things: the **CI widened** (τ² added real uncertainty) and the point estimate **moved toward the smaller studies**. The biggest trial (study 4) held **46%** of the fixed weight but only **33%** of the random weight — adding τ² to every denominator pulls the weights toward equal so no single trial silently dominates.

In [ ]:
_, _, _, wf = r['fixed']
_, _, _, wr = r['random']
print("study | fixed % | random %")
for i in range(4):
    print(f"  {i+1}   |  {wf[i]*100:4.1f}  |  {wr[i]*100:4.1f}")
print(f"\nstudy 4: {wf[3]*100:.0f}% -> {wr[3]*100:.0f}%  (the dominant trial loses influence)")

## Full readout vs the article's *Expected output*

In [ ]:
for model in ("fixed", "random"):
    est, lo, hi, weights = r[model]
    print(f"{model:6s}: OR = {np.exp(est):.3f}  "
          f"(95% CI {np.exp(lo):.3f} to {np.exp(hi):.3f})  "
          f"weights % = {np.round(weights*100, 1)}")
print(f"Q = {r['Q']:.2f}  df = {r['df']}  I2 = {r['I2']:.1f}%  tau2 = {r['tau2']:.4f}")

print("\n--- article Expected output ---")
print("fixed : OR = 0.706  (95% CI 0.578 to 0.862)  weights % = [25.9 16.6 11.5 46. ]")
print("random: OR = 0.658  (95% CI 0.478 to 0.906)  weights % = [27.1 22.  17.9 33. ]")
print("Q = 6.81  df = 3  I2 = 56.0%  tau2 = 0.0580")

## Recap

- **Inverse-variance weighting** (`w = 1/SE²`) gives precise studies more say — the minimum-variance combination.
- **Fixed-effect**: OR 0.71 (0.58–0.86), assumes one true effect.
- **Random-effects** (DerSimonian–Laird): OR 0.66 (0.48–0.91), adds **τ² = 0.058** so the CI widens and weights move toward equal.
- **Q = 6.81**, **I² = 56%** flagged moderate heterogeneity — the reason to prefer random-effects.
- Pool **ratios on the log scale**, then exponentiate; report the effect **with its CI** and show the weights.

← Back to the **Meta-Analysis** article. Next: **Heterogeneity & Forest Plots**, which reads Q, I², and τ² in full and draws the picture.